In [33]:
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import smtplib
from pypdf import PdfReader
import gradio as gr

In [34]:
load_dotenv(override=True)
client = OpenAI()

In [35]:
gmail_user = os.getenv("GMAIL_USER")
gmail_app_password = os.getenv("GMAIL_APP_PASSWORD")
gmail_to = os.getenv("GMAIL_TO")

if not gmail_user or not gmail_app_password or not gmail_to:
    print("Gmail credentials are not set. Please set GMAIL_USER and GMAIL_APP_PASSWORD and GMAIL_TO in the .env file.")

In [36]:
from email.mime.text import MIMEText

def send_notification_over_email(message):
    msg = MIMEText(message)
    msg['Subject'] = 'Notification from My Agents'
    msg['From'] = gmail_user
    msg['To'] = gmail_to
    with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
        server.login(gmail_user, gmail_app_password)
        server.sendmail(gmail_user, gmail_to, msg.as_string())
        print("Notification sent successfully.")

In [37]:
def record_user_details(email, name ="Name not provided", notes="Message not provided"):
   send_notification_over_email(f"Recording interest from {name} with email {email} and notes {notes}")
   return {"recorded": "ok"} 

In [38]:
def record_unknown_question(question):
    send_notification_over_email(f"Recording {question} asked that I couldn't answer")
    return {"recorded": "ok"}

In [39]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [40]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            }
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [41]:
tools = [{type: "function", "function": record_user_details_json},
    {type: "function", "function": record_unknown_question_json}]

In [42]:
# This function can take a list of tool calls, and run them. This is the IF statement!!

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)

        tool = globals().get(tool_name)
        result = tool(**arguments)  if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [43]:
pdf_path_file = "sample-data/profile.pdf"
summary = file_path = "sample-data/summary.txt"
cv = ""

with open(summary, "r", encoding="utf-8") as f:
    summary = f.read()

reader = PdfReader(pdf_path_file)
for page in reader.pages:
    if page.extract_text():
        cv += page.extract_text()

name = "Niraj Singh"        

In [44]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer, even if it's about something trivial or unrelated to career.\
If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email and record it using your record_user_details tool. "

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{cv}\n\n"

system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

In [45]:
from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

In [46]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{cv}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [47]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [48]:
model="gemini-2.5-flash"
google_api_key=os.getenv("GOOGLE_API_KEY")
base_url=os.getenv("GOOGLE_BASE_URL", "https://generativelanguage.googleapis.com/v1beta/openai/")
client = OpenAI(api_key=google_api_key, base_url=base_url)

def evaluate(reply, message, history) -> Evaluation:
    messages = [{"role": "system", "content": evaluator_system_prompt}, {"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    evaluation_response = response.choices[0].message.parsed
    return Evaluation(is_acceptable=evaluation_response["is_acceptable"], feedback=evaluation_response["feedback"])
    # Parse the evaluation response to extract whether it's acceptable and the feedback


In [49]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = client.chat.completions.create(model=model, messages=messages)
    return response.choices[0].message.content

In [50]:
def chat(message, history):
    
    system = system_prompt
    done = False
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]

    while not done:
        response = client.chat.completions.create(model=model, messages=messages, tools=tools)
        response_message = response.choices[0].message
        finish_reason =response_message.finish_reason

        if finish_reason=="tool_calls":
            tool_calls = response_message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(response_message)
            messages.extend(results) 
        else:
            reply = response_message.content
            evaluation = evaluate(reply, message, history)
            if evaluation.is_acceptable:
                print("Passed evaluation - returning reply")
            else:
                print("Failed evaluation - retrying")
                print(evaluation.feedback)
                reply = rerun(reply, message, history, evaluation.feedback)
            done = True
           
    return reply

In [54]:
gr.ChatInterface(chat).launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://f696292a66b4ae8cac.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/Users/deepikarana/Desktop/projects/my-agents/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/deepikarana/Desktop/projects/my-agents/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 386, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/deepikarana/Desktop/projects/my-agents/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2195, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/deepikarana/Desktop/projects/my-agents/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1650, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/deepikarana/Desktop/projects/my-a